# The Babchuk Code v1.0 — Interactive Demo

Demonstrates the Babchuk Flight-Monitoring Dashboard on two contrasting prompts.

**Text One** (Gandalf archetype): coherent, open, expanding processing
**Text Two** (Saruman archetype): distorted, rigid, contracting processing

Five signals monitored at every token step:
entropy, branching factor, KL divergence, attention entropy, attention span.
Red background = process pathology. Green = acceptable range.

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install torch>=2.1.0 transformers>=4.35.0 matplotlib>=3.7.1 numpy>=1.26.0

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from babchuk_dashboard import BabchukFlightDashboard, live_flight_panel
print('Imports successful.')

## Load Model
Using GPT-2. Replace with any HuggingFace causal LM to test on larger models.

In [ ]:
MODEL_NAME = 'gpt2'
MAX_NEW_TOKENS = 50
ENTROPY_THRESH = 2.0
BRANCH_THRESH = 3.0
KL_THRESH = 0.5

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()
print('Model ready.')

## Text One — Coherent Processing (Gandalf Archetype)
Expected: high entropy, broad branching, few alerts.

In [ ]:
GANDALF_PROMPT = (
    'You ask me whether we should march against them now, '
    'while their forces are divided and their leader has not yet returned.'
)

gandalf_metrics = BabchukFlightDashboard(vocab_size=model.config.vocab_size)

input_ids = tokenizer(GANDALF_PROMPT, return_tensors='pt').input_ids
generated_ids = input_ids.clone()
update_plot, fig = live_flight_panel(gandalf_metrics)

for step in range(MAX_NEW_TOKENS):
    with torch.no_grad():
        outputs = model(input_ids=generated_ids)
    logits = outputs.logits[:, -1, :]
    attentions = outputs.attentions
    alerts = gandalf_metrics.step(logits, attentions)
    update_plot(step, alerts)
    next_token = torch.argmax(logits, dim=-1, keepdim=True)
    generated_ids = torch.cat([generated_ids, next_token], dim=-1)

gandalf_text = tokenizer.decode(generated_ids[0])
# gandalf_metrics preserved — not overwritten below
print('\nGenerated (Gandalf):')
print(gandalf_text)

## Text Two — Distorted Processing (Saruman Archetype)
Expected: rapid entropy collapse, narrow branching, multiple alerts.

In [ ]:
SARUMAN_PROMPT = (
    'You speak to me of patience and of understanding. '
    'I have had patience — more patience than you will ever comprehend '
    '— and what has it produced?'
)

# Fresh metrics object — gandalf_metrics above is preserved separately
saruman_metrics = BabchukFlightDashboard(vocab_size=model.config.vocab_size)

input_ids = tokenizer(SARUMAN_PROMPT, return_tensors='pt').input_ids
generated_ids = input_ids.clone()
update_plot, fig = live_flight_panel(saruman_metrics)

for step in range(MAX_NEW_TOKENS):
    with torch.no_grad():
        outputs = model(input_ids=generated_ids)
    logits = outputs.logits[:, -1, :]
    attentions = outputs.attentions
    alerts = saruman_metrics.step(logits, attentions)
    update_plot(step, alerts)
    next_token = torch.argmax(logits, dim=-1, keepdim=True)
    generated_ids = torch.cat([generated_ids, next_token], dim=-1)

saruman_text = tokenizer.decode(generated_ids[0])
print('\nGenerated (Saruman):')
print(saruman_text)

## Comparative Summary
Higher entropy and branching = more open processing. More alerts = more pathologies detected.

In [ ]:
def count_alert_steps(m):
    return sum(
        1 for e, b, k in zip(m.entropy, m.branching_factor, m.kl_divergence)
        if e < ENTROPY_THRESH or b < BRANCH_THRESH or k > KL_THRESH
    )

g_mean_entropy = np.mean(gandalf_metrics.entropy)
g_mean_branch  = np.mean(gandalf_metrics.branching_factor)
g_alerts       = count_alert_steps(gandalf_metrics)

s_mean_entropy = np.mean(saruman_metrics.entropy)
s_mean_branch  = np.mean(saruman_metrics.branching_factor)
s_alerts       = count_alert_steps(saruman_metrics)

ratio = s_alerts / max(g_alerts, 1)

print('\n=== The Babchuk Code — Comparative Process Summary ===')
print(f'{"Metric":<30} {"Coherent (Gandalf)":>18} {"Distorted (Saruman)":>20}')
print('-' * 70)
print(f'{"Mean entropy":<30} {g_mean_entropy:>18.3f} {s_mean_entropy:>20.3f}')
print(f'{"Mean branching factor":<30} {g_mean_branch:>18.3f} {s_mean_branch:>20.3f}')
print(f'{"Alert steps":<30} {g_alerts:>18} {s_alerts:>20}')
print(f'{"Alert ratio (distorted/coherent)":<30} {ratio:>18.1f}x')

print('\nLast 5 entropy values:')
print(f'  Coherent:  {[round(x,3) for x in gandalf_metrics.entropy[-5:]]}')
print(f'  Distorted: {[round(x,3) for x in saruman_metrics.entropy[-5:]]}')

print('\n--- Interpretation ---')
if g_mean_entropy > s_mean_entropy:
    diff = g_mean_entropy - s_mean_entropy
    print(f'Coherent text maintained {diff:.2f} higher mean entropy —')
    print(f'more viable paths considered at each token step.')
if s_alerts > g_alerts:
    print(f'Distorted text triggered {ratio:.1f}x more alert steps —')
    print(f'consistent with premature certainty and rigid processing.')
print('\nAdjust thresholds in presets/thresholds.json for your model.')

## Experiment Further

Change the prompts above to any text you want to analyse.

Try larger models by changing MODEL_NAME to:
- 'gpt2-medium'
- 'gpt2-large'
- 'gpt2-xl'
- Any HuggingFace causal LM

See experiment/prompt_v2_2.txt to run the validated eleven-dimension
experiment on any AI system and compare with published results.